# Bài 5.3 · Augmentation và transfer learning khi ít dữ liệu

**Câu hỏi của buổi này:** chỉ có 2 000 ảnh có nhãn thì làm sao để model không học thuộc? Hai công cụ: **augmentation** (biến đổi ảnh mà nhãn không đổi) và **transfer learning** (mượn encoder đã học trên ImageNet).

Dữ liệu là **CIFAR-10**, nhưng ta chỉ lấy **200 ảnh mỗi lớp** để train; 2 000 ảnh test để chấm.

| Mục | Sẽ thấy |
|---|---|
| 1. Dữ liệu | bảng hai tập |
| 2. Augmentation trông thế nào | một ảnh qua 8 lần biến đổi ngẫu nhiên; chữ số bị lật thì đổi nghĩa |
| 3. Train từ đầu: có và không augmentation | đường accuracy train/test của hai cách |
| 4. Transfer learning | bảng và biểu đồ năm cách: từ đầu, linear probe, fine-tune |
| 5. Fine-tune đổi tầng nào | mức thay đổi trọng số theo từng giai đoạn của ResNet-18 |
| 6. Ảnh xám vào encoder ba kênh | accuracy ảnh màu và ảnh xám; kiểm cách gộp bộ lọc |
| 7. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **GPU** và **Internet** trong *Session options*; chạy hết khoảng 6 phút. Bài giảng: [ai-canuc80k · bài 5.3](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/5.3).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style
    if fmt:
        st = st.format(fmt)
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import time, copy, random
import torch, torchvision
from torch import nn
import torch.nn.functional as F
from torchvision.transforms import v2 as T
import datasets
from datasets import load_dataset
datasets.disable_progress_bars()

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

seed_all(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
PER_CLASS, N_TEST, IMG = 200, 2000, 160
EP_SMALL, EP_BIG = 40, 12
table(pd.DataFrame([('torch', torch.__version__), ('torchvision', torchvision.__version__),
                    ('thiết bị', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. Dữ liệu

In [ ]:
with quiet():
    cifar = load_dataset('uoft-cs/cifar10')
VI = {'airplane': 'máy bay', 'automobile': 'ô tô', 'bird': 'chim', 'cat': 'mèo', 'deer': 'hươu',
      'dog': 'chó', 'frog': 'ếch', 'horse': 'ngựa', 'ship': 'tàu thủy', 'truck': 'xe tải'}
vi = [VI[c] for c in cifar['train'].features['label'].names]

labels = np.array(cifar['train']['label'])
tr_rows = np.concatenate([np.flatnonzero(labels == k)[:PER_CLASS] for k in range(10)])
def to_tensor(split, rows):
    d = split.select(rows)
    x = torch.from_numpy(np.stack([np.asarray(im.convert('RGB')) for im in d['img']])).permute(0, 3, 1, 2).contiguous()
    return x.to(device), torch.tensor(d['label']).to(device)
X_tr8, Y_tr = to_tensor(cifar['train'], tr_rows)
X_te8, Y_te = to_tensor(cifar['test'], range(N_TEST))
table(pd.DataFrame([('train', len(Y_tr), f'{PER_CLASS} ảnh mỗi lớp'), ('test', len(Y_te), 'chấm sau mỗi epoch')], columns=['tập', 'số ảnh', 'ghi chú']), 'Hai tập')
check('train cân bằng: mỗi lớp đúng số ảnh', (torch.bincount(Y_tr) == PER_CLASS).all().item())

## 2. Augmentation trông thế nào

Mỗi lần ảnh được lấy ra để train, nó bị biến đổi ngẫu nhiên một chút: cắt một khung lệch vài pixel rồi đệm viền, lật ngang, đổi nhẹ độ sáng và màu. Ảnh vẫn là con vật ấy, nên nhãn giữ nguyên.

In [ ]:
augment = T.Compose([T.RandomCrop(32, padding=4, padding_mode='reflect'), T.RandomHorizontalFlip(),
                     T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3)])
i = int(torch.nonzero(Y_tr == vi.index('ngựa')).flatten()[3])
torch.manual_seed(1)
fig, axes = plt.subplots(1, 9, figsize=(15, 2.2))
axes[0].imshow(X_tr8[i].permute(1, 2, 0).cpu()); axes[0].set_title('ảnh gốc', fontsize=10)
for ax in axes[1:]:
    ax.imshow(augment(X_tr8[i]).permute(1, 2, 0).cpu()); ax.set_title('biến đổi', fontsize=10, color=C['gray'])
for ax in axes: ax.axis('off')
fig.suptitle('Một ảnh ngựa, tám lần lấy ra là tám ảnh hơi khác nhau', y=1.08)
plt.tight_layout(); plt.show()

Không phải phép biến đổi nào cũng giữ nhãn. Với chữ số viết tay, **lật ngang** biến nhiều chữ số thành hình không còn là chữ số ấy.

In [ ]:
with quiet():
    mnist = torchvision.datasets.MNIST('/tmp/mnist', train=False, download=True)
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for c, d in enumerate([2, 3, 4, 5, 6, 7, 9, 1]):
    img = mnist.data[(mnist.targets == d).nonzero()[0, 0]]
    axes[0, c].imshow(img, cmap='gray_r'); axes[0, c].set_title(f'chữ {d}', fontsize=10)
    axes[1, c].imshow(img.flip(1), cmap='gray_r'); axes[1, c].set_title('lật ngang', fontsize=10, color=C['clay'])
for ax in axes.flat: ax.axis('off')
fig.suptitle('Lật ngang giữ nhãn cho ảnh con vật, nhưng phá nhãn của chữ số (trừ những số đối xứng như 1)', y=1.04)
plt.tight_layout(); plt.show()

## 3. Train từ đầu: có và không augmentation

Cùng một CNN nhỏ (ba block conv), cùng số epoch. Chỉ khác: một bên thấy đúng 2 000 ảnh gốc mỗi epoch, một bên thấy 2 000 ảnh đã biến đổi ngẫu nhiên.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device)[:, None, None]
STD = torch.tensor([0.229, 0.224, 0.225], device=device)[:, None, None]
def prep(x8, size=None):
    x = (x8.float() / 255 - MEAN) / STD
    return F.interpolate(x, size=size, mode='bilinear', align_corners=False) if size else x

def small_cnn():
    def block(cin, cout):
        return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(),
                             nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(), nn.MaxPool2d(2))
    return nn.Sequential(block(3, 32), block(32, 64), block(64, 128), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, 10)).to(device)

def accuracy(model, X8, Y, size=None, bs=500):
    model.eval(); ok = 0
    with torch.no_grad():
        for i in range(0, len(Y), bs):
            ok += (model(prep(X8[i:i + bs], size)).argmax(1) == Y[i:i + bs]).sum().item()
    return ok / len(Y)

def train(model, epochs, lr, aug, size=None, params=None, frozen=False, bs=128):
    opt = torch.optim.AdamW(params if params is not None else model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * ((len(Y_tr) + bs - 1) // bs))
    gen, hist = torch.Generator().manual_seed(0), []
    for ep in range(1, epochs + 1):
        model.train(not frozen); ok = 0
        for idx in torch.randperm(len(Y_tr), generator=gen).to(device).split(bs):
            x8 = torch.stack([augment(x) for x in X_tr8[idx]]) if aug else X_tr8[idx]
            out = model(prep(x8, size)); loss = F.cross_entropy(out, Y_tr[idx])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            ok += (out.argmax(1) == Y_tr[idx]).sum().item()
        hist.append((ep, ok / len(Y_tr), accuracy(model, X_te8, Y_te, size)))
    return pd.DataFrame(hist, columns=['epoch', 'train acc', 'test acc'])

runs, t0 = {}, time.time()
for name, aug in [('không augmentation', False), ('có augmentation', True)]:
    seed_all(0); runs[name] = train(small_cnn(), EP_SMALL, 3e-3, aug)

fig, ax = plt.subplots(figsize=(10, 4.2))
for (name, h), c in zip(runs.items(), [C['clay'], C['blue']]):
    ax.plot(h['epoch'], h['train acc'], color=c, ls='--', lw=1.5, label=f'{name}: train')
    ax.plot(h['epoch'], h['test acc'], color=c, lw=2.5, label=f'{name}: test')
ax.set(xlabel='epoch', ylabel='accuracy', title=f'CNN nhỏ train từ đầu trên {len(Y_tr)} ảnh ({time.time() - t0:.0f} giây)', ylim=(0, 1.02))
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout(); plt.show()
scratch = {n: h.iloc[-1] for n, h in runs.items()}
table(pd.DataFrame([(n, r['train acc'], r['test acc'], r['train acc'] - r['test acc']) for n, r in scratch.items()],
                   columns=['cách', 'train acc', 'test acc', 'khoảng cách']), f'Sau {EP_SMALL} epoch', fmt={c: '{:.3f}' for c in ['train acc', 'test acc', 'khoảng cách']}, best='test acc')
a0, a1 = scratch['không augmentation'], scratch['có augmentation']
note(f"Không augmentation: train <b>{a0['train acc']:.2f}</b> mà test chỉ {a0['test acc']:.2f}, model học thuộc {len(Y_tr):,} ảnh. "
     f"Có augmentation: test {a1['test acc']:.2f} ({a1['test acc'] - a0['test acc']:+.2f}), khoảng cách train–test hẹp lại.")

## 4. Transfer learning

ResNet-18 đã học trên 1.2 triệu ảnh ImageNet. Ảnh CIFAR được phóng lên 160×160 cho gần cỡ ảnh nó từng thấy. Ba cách dùng, so với hai cách train từ đầu ở trên:

- **ResNet-18 từ đầu**: cùng kiến trúc, trọng số ngẫu nhiên;
- **linear probe**: đóng băng toàn bộ encoder, chỉ học lớp Linear cuối;
- **fine-tune**: học tiếp mọi tầng với learning rate nhỏ.

Cả ba đều có augmentation.

In [ ]:
W = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
def resnet(pretrained):
    with quiet():
        m = torchvision.models.resnet18(weights=W if pretrained else None)
    m.fc = nn.Linear(512, 10)
    return m.to(device)

big, t0 = {}, time.time()
seed_all(0); big['ResNet-18 từ đầu'] = train(resnet(False), EP_BIG, 2e-3, True, IMG)
seed_all(0); probe = resnet(True); probe.requires_grad_(False); probe.fc.requires_grad_(True)
big['linear probe'] = train(probe, EP_BIG, 3e-3, True, IMG, params=probe.fc.parameters(), frozen=True)
seed_all(0); ft = resnet(True); w0 = copy.deepcopy(ft.state_dict())
big['fine-tune'] = train(ft, EP_BIG, 5e-4, True, IMG)

res = pd.DataFrame([('CNN nhỏ từ đầu, không augmentation', a0['test acc']), ('CNN nhỏ từ đầu, có augmentation', a1['test acc'])] +
                   [(n, h['test acc'].iloc[-1]) for n, h in big.items()], columns=['cách', 'test acc'])
table(res, f'Năm cách, cùng {len(Y_tr)} ảnh train ({time.time() - t0:.0f} giây cho ba cách ResNet)', fmt={'test acc': '{:.3f}'}, best='test acc')

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={'width_ratios': [1.1, 1]})
cols = [C['gray'], C['gray'], C['sand'], C['blue'], C['green']]
axes[0].barh(res['cách'], res['test acc'], color=cols); axes[0].invert_yaxis(); axes[0].set_xlim(0, 1.05)
axes[0].set_title('Test accuracy'); axes[0].grid(axis='y', visible=False); bar_labels(axes[0], '{:.3f}')
for (n, h), c in zip(big.items(), cols[2:]):
    axes[1].plot(h['epoch'], h['test acc'], marker='o', ms=3, color=c, label=n)
axes[1].set(xlabel='epoch', ylabel='test accuracy', title='Ba cách dùng ResNet-18 theo epoch', ylim=(0, 1)); axes[1].legend()
plt.tight_layout(); plt.show()
best = res.loc[res['test acc'].idxmax()]
check('fine-tune vượt ResNet-18 train từ đầu', big['fine-tune']['test acc'].iloc[-1] > big['ResNet-18 từ đầu']['test acc'].iloc[-1])
note(f"Cao nhất: <b>{best['cách']}</b> với {best['test acc']:.3f}. Linear probe chỉ học {10 * 513:,} tham số mà đạt {big['linear probe']['test acc'].iloc[-1]:.3f}; "
     f"cùng kiến trúc nhưng train từ đầu chỉ được {big['ResNet-18 từ đầu']['test acc'].iloc[-1]:.3f}. Thứ được chuyển sang là các bộ lọc đã học trên ImageNet.")

## 5. Fine-tune đổi tầng nào nhiều nhất

So trọng số sau fine-tune với trọng số ImageNet ban đầu: mỗi giai đoạn đổi bao nhiêu phần trăm (độ lớn phần thay đổi chia độ lớn ban đầu).

In [ ]:
w1 = ft.state_dict()
groups = ['conv1', 'layer1', 'layer2', 'layer3', 'layer4']
change = []
for g in groups:
    keys = [k for k in w0 if k.startswith(g + '.') and k.endswith('weight') and w0[k].dim() == 4]
    d = torch.sqrt(sum(((w1[k] - w0[k]).float() ** 2).sum() for k in keys)); n = torch.sqrt(sum((w0[k].float() ** 2).sum() for k in keys))
    change.append((g, (d / n).item()))
chg = pd.DataFrame(change, columns=['giai đoạn', 'thay đổi tương đối'])
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(chg['giai đoạn'], chg['thay đổi tương đối'] * 100, color=C['blue']); bar_labels(ax, '{:.1f}%')
ax.set_ylabel('% thay đổi'); ax.set_title('Fine-tune: tầng gần ảnh đổi ít, tầng sâu đổi nhiều hơn')
plt.tight_layout(); plt.show()
note(f"conv1 đổi {chg['thay đổi tương đối'].iloc[0]:.1%}, layer4 đổi <b>{chg['thay đổi tương đối'].iloc[-1]:.1%}</b>. "
     f"Bộ lọc nét và màu ở tầng đầu dùng được cho mọi ảnh, nên gần như giữ nguyên.")

## 6. Ảnh xám vào encoder ba kênh

Encoder ImageNet nhận ảnh ba kênh. Có ảnh xám (X-quang, chữ viết tay) thì có hai cách tương đương:

- lặp kênh xám thành ba kênh giống nhau;
- cộng ba bộ lọc màu của conv đầu thành một bộ lọc cho ảnh một kênh.

Cách thứ hai cho đúng cùng kết quả mà nhẹ hơn. Sau đó so linear probe trên ảnh màu và ảnh xám.

In [ ]:
conv = probe.conv1
g = torch.rand(4, 1, 64, 64, device=device)
one = F.conv2d(g, conv.weight.sum(1, keepdim=True), stride=2, padding=3)
three = conv(g.repeat(1, 3, 1, 1))
check(f'cộng bộ lọc = lặp kênh (chênh lớn nhất {(one - three).abs().max().item():.1e})', torch.allclose(one, three, atol=1e-4))

gray = lambda x8: (0.299 * x8[:, 0] + 0.587 * x8[:, 1] + 0.114 * x8[:, 2]).round().clamp(0, 255).to(torch.uint8)[:, None].repeat(1, 3, 1, 1)
X_tr8_color, X_te8_color = X_tr8, X_te8
X_tr8, X_te8 = gray(X_tr8_color), gray(X_te8_color)
seed_all(0); probe_g = resnet(True); probe_g.requires_grad_(False); probe_g.fc.requires_grad_(True)
h_gray = train(probe_g, EP_BIG, 3e-3, True, IMG, params=probe_g.fc.parameters(), frozen=True)
X_tr8, X_te8 = X_tr8_color, X_te8_color

fig, axes = plt.subplots(1, 6, figsize=(12, 2.3))
for c, name in enumerate(['ô tô', 'mèo', 'ếch']):
    j = int(torch.nonzero(Y_te == vi.index(name)).flatten()[0])
    axes[2 * c].imshow(X_te8[j].permute(1, 2, 0).cpu()); axes[2 * c].set_title(name, fontsize=10)
    axes[2 * c + 1].imshow(gray(X_te8[j:j + 1])[0].permute(1, 2, 0).cpu()); axes[2 * c + 1].set_title(f'{name} (xám)', fontsize=10)
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()
color_acc, gray_acc = big['linear probe']['test acc'].iloc[-1], h_gray['test acc'].iloc[-1]
table(pd.DataFrame([('ảnh màu', color_acc), ('ảnh xám lặp ba kênh', gray_acc)], columns=['đầu vào', 'test acc (linear probe)']), 'Mất màu thì mất bao nhiêu', fmt={'test acc (linear probe)': '{:.3f}'})
note(f'Bỏ màu làm linear probe giảm <b>{color_acc - gray_acc:.3f}</b>; encoder vẫn dùng được cho ảnh xám nhờ nét và hình dạng.')

## 7. Tóm tắt

In [ ]:
summary = [('CNN nhỏ, không augmentation: train / test', f"{a0['train acc']:.3f} / {a0['test acc']:.3f}"),
           ('CNN nhỏ, có augmentation: train / test', f"{a1['train acc']:.3f} / {a1['test acc']:.3f}"),
           ('ResNet-18 từ đầu', f"{big['ResNet-18 từ đầu']['test acc'].iloc[-1]:.3f}"),
           ('ResNet-18 ImageNet, linear probe', f'{color_acc:.3f}'),
           ('ResNet-18 ImageNet, fine-tune', f"{big['fine-tune']['test acc'].iloc[-1]:.3f}"),
           ('linear probe trên ảnh xám', f'{gray_acc:.3f}')]
table(pd.DataFrame(summary, columns=['điều đo (test accuracy)', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** đổi `PER_CLASS = 200` thành 50 hay 1000 để xem khoảng cách giữa train từ đầu và transfer co giãn ra sao; bỏ `ColorJitter` khỏi augmentation; thêm `T.RandomVerticalFlip()` và xem nó có giúp ảnh CIFAR không.